# Full end-to-end benchmark — `is_smiling()`
Measures the production entry point `is_smiling(img)` (SCRFD detection with
zoom-crop fallback → geometric features → fitted scaler → `LogisticRegression`)
on the **full test split**, i.e. the clean 794 single-face test images *plus*
the 29 images that were excluded from `dataset_clean.pkl` (0-face and multi-face)
with their ground-truth labels.

Each call is timed individually with the model/detector already loaded (warmed
once before the loop). Latency is compared against the 4.7 ms Stage-4 baseline.
Results are saved to `outputs/benchmark_results.csv`.

In [1]:
import sys, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

PROJECT_ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
OUT_DIR = PROJECT_ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from smile_detector import is_smiling, get_detector

LATENCY_BASELINE_MS = 4.7

features = pd.read_pickle(CACHE_DIR / "features.pkl")
raw = pd.read_pickle(CACHE_DIR / "detections_raw.pkl")
print(f"features.pkl (clean single-face): {len(features)} rows")
print(f"detections_raw.pkl (all images) : {len(raw)} rows")

features.pkl (clean single-face): 3971 rows
detections_raw.pkl (all images) : 3999 rows


In [2]:
from sklearn.model_selection import train_test_split

# Reproduce the exact stratified 80/20 split from 04_modeling.ipynb
# (same input order, same random_state -> identical test filenames).
df_train, df_test_clean = train_test_split(
    features, test_size=0.2, stratify=features["label"], random_state=42
)

# The images excluded from dataset_clean.pkl / features.pkl (0-face + multi-face).
excluded = raw[raw["n_faces"] != 1].copy()
print(f"clean test (single-face)  : {len(df_test_clean)}")
print(f"excluded (0-face/multi)    : {len(excluded)} ({excluded['label'].value_counts().sort_index().to_dict()})")

full_test = pd.concat([
    df_test_clean[["filename", "filepath", "label"]].assign(n_faces=1),
    excluded[["filename", "filepath", "label", "n_faces"]],
], ignore_index=True)
full_test = full_test.reset_index(drop=True)
print(f"FULL test set: {len(full_test)} images")
print(full_test["label"].value_counts().sort_index().rename({0: "non_smile", 1: "smile"}))

clean test (single-face)  : 795
excluded (0-face/multi)    : 28 ({0: 8, 1: 20})
FULL test set: 823 images
label
non_smile    372
smile        451
Name: count, dtype: int64


In [3]:
# Preload every image once so the timed loop measures is_smiling() only
# (no disk I/O inside the timed section).
images = []
for fp in full_test["filepath"]:
    img = cv2.imread(fp)
    images.append(img)
missing = sum(img is None for img in images)
print(f"preloaded {len(images)} images, {missing} unreadable")

# Warm-up: load the detector once and run one inference before the loop so
# model/detector load and ORT warm-up are excluded from the timings.
_ = get_detector()
_ = is_smiling(images[0])
print("detector + model warmed")

preloaded 823 images, 0 unreadable
detector + model warmed


In [4]:
latencies_ms = []
predictions = []
for img in images:
    t0 = time.perf_counter()
    pred = is_smiling(img)
    t1 = time.perf_counter()
    predictions.append(int(pred))
    latencies_ms.append((t1 - t0) * 1e3)

full_test["prediction"] = predictions
full_test["latency_ms"] = latencies_ms
print("benchmark loop done")

benchmark loop done


In [5]:
y_true = full_test["label"].to_numpy()
y_pred = full_test["prediction"].to_numpy()

print("=== Full test-set metrics (823 images incl. 0-face / multi-face) ===")
print(f"accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"precision: {precision_score(y_true, y_pred):.4f}")
print(f"recall   : {recall_score(y_true, y_pred):.4f}")
print(f"F1       : {f1_score(y_true, y_pred):.4f}")
print()
print("Confusion matrix (rows=true, cols=predicted, [[TN FP],[FN TP]]):")
cm = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(cm, index=["true non_smile", "true smile"], columns=["pred non_smile", "pred smile"]).to_string())

=== Full test-set metrics (823 images incl. 0-face / multi-face) ===
accuracy : 0.8518
precision: 0.9387
recall   : 0.7805
F1       : 0.8523

Confusion matrix (rows=true, cols=predicted, [[TN FP],[FN TP]]):
                pred non_smile  pred smile
true non_smile             349          23
true smile                  99         352


In [6]:
lat = np.asarray(full_test["latency_ms"])

med = float(np.median(lat))
p95 = float(np.percentile(lat, 95))
mean = float(lat.mean())

print("=== Per-call latency (model/detector load excluded) ===")
print(f"mean   : {mean:.3f} ms")
print(f"median : {med:.3f} ms")
print(f"p95    : {p95:.3f} ms")
print(f"min/max: {lat.min():.3f} / {lat.max():.3f} ms")
print()
print(f"Baseline: {LATENCY_BASELINE_MS} ms (eKYC Stage-4 budget)")
delta_med = med - LATENCY_BASELINE_MS
delta_p95 = p95 - LATENCY_BASELINE_MS
print(f"median {med:.3f} ms {'MEETS' if delta_med <= 0 else 'EXCEEDS'} baseline "
      f"(delta {delta_med:+.3f} ms)")
print(f"p95    {p95:.3f} ms {'MEETS' if delta_p95 <= 0 else 'EXCEEDS'} baseline "
      f"(delta {delta_p95:+.3f} ms)")

# Primary verdict on the median (typical call); flag the p95 tail explicitly.
met = med <= LATENCY_BASELINE_MS
print("\n==> LATENCY BUDGET MET." if met else "\n==> LATENCY BUDGET NOT MET.")
if met and p95 > LATENCY_BASELINE_MS:
    print(f"    Caveat: median meets it, but p95 ({p95:.3f} ms) is {abs(delta_p95):.3f} ms above "
          f"the budget - the tail of hard fallback images pushes ~5% of calls over.")

=== Per-call latency (model/detector load excluded) ===
mean   : 3.699 ms
median : 3.573 ms
p95    : 5.149 ms
min/max: 1.916 / 18.302 ms

Baseline: 4.7 ms (eKYC Stage-4 budget)
median 3.573 ms MEETS baseline (delta -1.127 ms)
p95    5.149 ms EXCEEDS baseline (delta +0.449 ms)

==> LATENCY BUDGET MET.
    Caveat: median meets it, but p95 (5.149 ms) is 0.449 ms above the budget - the tail of hard fallback images pushes ~5% of calls over.


In [7]:
result = full_test[["filename", "filepath", "n_faces", "label", "prediction", "latency_ms"]]
result.to_csv(OUT_DIR / "benchmark_results.csv", index=False)

summary = {
    "n_images": int(len(result)),
    "n_excluded_hard": int((result["n_faces"] != 1).sum()),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred),
    "recall": recall_score(y_true, y_pred),
    "f1": f1_score(y_true, y_pred),
    "latency_mean_ms": mean,
    "latency_median_ms": med,
    "latency_p95_ms": p95,
    "baseline_latency_ms": LATENCY_BASELINE_MS,
    "latency_met": bool(met),
}
pd.Series(summary).to_csv(OUT_DIR / "benchmark_summary.csv", header=False)

print(f"Saved per-image results ({len(result)} rows) -> outputs/benchmark_results.csv")
print("Saved metric/latency summary -> outputs/benchmark_summary.csv")
assert len(result) == 823, f"expected 823 rows, got {len(result)}"
assert result["latency_ms"].notna().all()
print("OK")

Saved per-image results (823 rows) -> outputs/benchmark_results.csv
Saved metric/latency summary -> outputs/benchmark_summary.csv
OK
